# PS LiDAR - Laboratorio de Desarrollo

**Ladrillos disponibles:**
- Brick 1: Carga de Datos
- Brick 2: Recorte Circular (coordenadas manuales)
- Brick 3: Detección de Normalización
- Brick 4: Filtrado de Suelo
- Brick 5: Normalización de Altura
- Brick 5.5: Exportar Checkpoints
- Brick 6: Visualización 3D
- **Brick 7: Segmentación de Árboles** ← NUEVO

In [1]:
import os
import sys
import time
from pathlib import Path

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.core import (
    PointCloudLoader,
    detect_normalization,
    classify_ground,
    clip_circular_plot,
    normalize_heights,
    export_point_cloud,
    segment_trees,
)

print("✓ Módulos importados")

✓ Módulos importados


---
## 1. Cargar Archivo (Brick 1)

In [2]:
FILE_PATH = "C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/raw/HQP080_01_raw.laz"

loader = PointCloudLoader(FILE_PATH)
loader.load()

meta = loader.get_metadata()
print(f"Archivo: {meta['filename']}")
print(f"Puntos: {meta['point_count']:,}")
print(f"Tamaño: {meta['file_size_mb']} MB")

Archivo: HQP080_01_raw.laz
Puntos: 23,934,061
Tamaño: 258.2 MB


In [3]:
# Cargar XYZ y campos escalares disponibles
xyz_full = loader.get_xyz()

scalar_fields = {}
for field in ['intensity', 'return_number', 'number_of_returns', 'classification']:
    try:
        scalar_fields[field] = loader.get_attribute(field)
        print(f"✓ {field}: {len(scalar_fields[field]):,} valores")
    except:
        print(f"⚠ {field}: no disponible")

print(f"\nMemoria XYZ: {xyz_full.nbytes / (1024**2):.1f} MB")

✓ intensity: 23,934,061 valores
✓ return_number: 23,934,061 valores
✓ number_of_returns: 23,934,061 valores
✓ classification: 23,934,061 valores

Memoria XYZ: 547.8 MB


---
## 2. Recorte Circular (Brick 2)

**Instrucciones:**
1. Abrir el archivo original en CloudCompare
2. Usar herramienta "Point Picking" para ubicar el centro del mat
3. Copiar las coordenadas Xg, Yg mostradas
4. Pegar los valores en `CENTER_X` y `CENTER_Y` abajo

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PARÁMETROS DEL USUARIO - Modificar según el plot
# ═══════════════════════════════════════════════════════════════

# Coordenadas del centro (obtenidas de CloudCompare Point Picking)
CENTER_X = -0.809949
CENTER_Y = -0.745654

# Radio del plot en metros
PLOT_RADIUS = 16.0

# ═══════════════════════════════════════════════════════════════

print(f"Centro: ({CENTER_X:.6f}, {CENTER_Y:.6f})")
print(f"Radio: {PLOT_RADIUS}m")

In [ ]:
# Ejecutar recorte circular
t0 = time.perf_counter()
clip_result = clip_circular_plot(xyz_full, CENTER_X, CENTER_Y, PLOT_RADIUS)
elapsed = time.perf_counter() - t0

plot_indices = clip_result.indices

print(f"✓ Recorte en {elapsed*1000:.0f}ms")
print(f"Puntos originales: {len(xyz_full):,}")
print(f"Puntos en plot: {clip_result.n_points:,} ({clip_result.n_points/len(xyz_full):.1%})")

In [ ]:
# Aplicar recorte a XYZ y campos escalares
xyz = xyz_full[plot_indices]

plot_scalars = {}
for field, values in scalar_fields.items():
    plot_scalars[field] = values[plot_indices]

print(f"Plot XYZ: {xyz.shape}")
print(f"Campos escalares: {list(plot_scalars.keys())}")

# Liberar memoria
del xyz_full, scalar_fields
import gc; gc.collect()
print("✓ Memoria liberada")

---
## 3. Análisis de Normalización (Brick 3)

In [ ]:
analysis = detect_normalization(xyz)
print(f"Estatus: {analysis.status.value.upper()}")
print(f"¿Normalizada?: {analysis.is_normalized}")
print(f"Rango Z: {analysis.z_min:.2f}m a {analysis.z_max:.2f}m")

---
## 4. Filtrado de Suelo (Brick 4)

In [ ]:
print("Ejecutando CSF...")
t0 = time.perf_counter()

ground_result = classify_ground(
    xyz,
    cloth_resolution=1.0,
    rigidness=1,
    class_threshold=0.5,
    slope_smooth=True,
)

print(f"✓ Completado en {time.perf_counter() - t0:.2f}s")
print(f"Suelo: {ground_result.n_ground:,} ({ground_result.ground_ratio:.1%})")
print(f"Vegetación: {ground_result.n_off_ground:,}")

In [ ]:
# Separar suelo y vegetación
ground_xyz = xyz[ground_result.ground_indices]
vegetation_xyz = xyz[ground_result.off_ground_indices]

ground_scalars = {k: v[ground_result.ground_indices] for k, v in plot_scalars.items()}
vegetation_scalars = {k: v[ground_result.off_ground_indices] for k, v in plot_scalars.items()}

print(f"Suelo: {len(ground_xyz):,} puntos")
print(f"Vegetación: {len(vegetation_xyz):,} puntos")

---
## 5. Normalización de Altura (Brick 5)

In [ ]:
print("Normalizando alturas...")
t0 = time.perf_counter()

veg_norm_result = normalize_heights(vegetation_xyz, ground_xyz, resolution=0.5)
veg_normalized = veg_norm_result.xyz_normalized

ground_norm_result = normalize_heights(ground_xyz, ground_xyz, resolution=0.5)
ground_normalized = ground_norm_result.xyz_normalized

print(f"✓ Completado en {(time.perf_counter() - t0)*1000:.0f}ms")
print(f"")
print(f"Vegetación: Z = {veg_normalized[:, 2].min():.2f}m a {veg_normalized[:, 2].max():.2f}m")
print(f"Suelo: Z = {ground_normalized[:, 2].min():.2f}m a {ground_normalized[:, 2].max():.2f}m")

---
## 5.5 Exportar Checkpoints

In [2]:
# Directorio de salida
OUTPUT_DIR = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/raw")

# Exportar vegetación
veg_file = OUTPUT_DIR / "vegetation_normalized.laz"
export_point_cloud(
    veg_file,
    veg_normalized,
    intensity=vegetation_scalars.get('intensity'),
    return_number=vegetation_scalars.get('return_number'),
    number_of_returns=vegetation_scalars.get('number_of_returns'),
    classification=vegetation_scalars.get('classification'),
)
print(f"✓ Vegetación: {veg_file.name} ({veg_file.stat().st_size / (1024**2):.1f} MB)")

# Exportar suelo
ground_file = OUTPUT_DIR / "ground_normalized.laz"
export_point_cloud(
    ground_file,
    ground_normalized,
    intensity=ground_scalars.get('intensity'),
    return_number=ground_scalars.get('return_number'),
    number_of_returns=ground_scalars.get('number_of_returns'),
    classification=ground_scalars.get('classification'),
)
print(f"✓ Suelo: {ground_file.name} ({ground_file.stat().st_size / (1024**2):.1f} MB)")

NameError: name 'veg_normalized' is not defined

---
## 6. Visualización 3D (Brick 6)

In [ ]:
import open3d as o3d
import numpy as np

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(veg_normalized)

# Colorear por altura
z = veg_normalized[:, 2]
z_scaled = (z - z.min()) / (z.max() - z.min() + 1e-6)
colors = np.zeros((len(z_scaled), 3))
colors[:, 0] = z_scaled
colors[:, 1] = 1 - np.abs(2 * z_scaled - 1)
colors[:, 2] = 1 - z_scaled
pcd.colors = o3d.utility.Vector3dVector(colors)

print(f"Nube: {len(pcd.points):,} puntos")

In [ ]:
o3d.visualization.draw_geometries([pcd], window_name="Vegetación Normalizada", width=1280, height=720)

---
---
# BRICK 7: Segmentación de Árboles

**Opción A:** Continuar desde Brick 5 (si ya ejecutaste todo arriba)  
**Opción B:** Cargar checkpoint de vegetación normalizada (si reinicias kernel)

### Opción B: Cargar desde Checkpoint

Ejecuta esta celda SOLO si reiniciaste el kernel y quieres continuar desde el checkpoint.

In [3]:
# ═══════════════════════════════════════════════════════════════
# Instalar pgeof desde dentro del notebook
# ═══════════════════════════════════════════════════════════════

import sys
print(f"Python: {sys.executable}")
!{sys.executable} -m pip install pgeof

Python: c:\Users\geoal\Documents\SoftwareDev\PS_LiDAR\venv\Scripts\python.exe


In [5]:
# ═══════════════════════════════════════════════════════════════
# CARGAR DESDE CHECKPOINT (autónomo - incluye todos los imports)
# ═══════════════════════════════════════════════════════════════
import os
import sys
import time
import laspy
import numpy as np
from pathlib import Path

# Agregar módulo al path
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.core import segment_trees

CHECKPOINT_FILE = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/raw/vegetation_normalized_3.laz")

if CHECKPOINT_FILE.exists():
    print(f"Cargando: {CHECKPOINT_FILE.name}")
    las = laspy.read(str(CHECKPOINT_FILE))
    veg_normalized = np.column_stack([las.x, las.y, las.z])
    print(f"✓ Cargados {len(veg_normalized):,} puntos")
    print(f"Rango Z: {veg_normalized[:, 2].min():.2f}m a {veg_normalized[:, 2].max():.2f}m")
    print(f"✓ segment_trees importado")
else:
    print(f"⚠ Archivo no encontrado: {CHECKPOINT_FILE}")
    print("  Ejecuta los Bricks 1-5 primero.")

Cargando: vegetation_normalized_3.laz
✓ Cargados 11,620,517 puntos
Rango Z: -0.93m a 19.58m
✓ segment_trees importado


### 7.1 Segmentación de Árboles

In [7]:
# ═══════════════════════════════════════════════════════════════
# PARÁMETROS DE SEGMENTACIÓN
# ═══════════════════════════════════════════════════════════════

VOXEL_RESOLUTION = 0.05       # Resolución de voxelización (metros)
STRIPE_Z_MIN = 2.0          # Altura mínima para detección de tallos
STRIPE_Z_MAX = 7.0            # Altura máxima para detección de tallos
VERTICALITY_THRESHOLD = 0.7   # Umbral de verticalidad (0-1)
MAX_AXIS_DISTANCE = 2.0       # Distancia máxima al eje para asignación

# ═══════════════════════════════════════════════════════════════

In [8]:
print("Segmentando árboles...")
t0 = time.perf_counter()

seg_result = segment_trees(
    veg_normalized,
    voxel_resolution=VOXEL_RESOLUTION,
    stripe_z_min=STRIPE_Z_MIN,
    stripe_z_max=STRIPE_Z_MAX,
    verticality_threshold=VERTICALITY_THRESHOLD,
    max_axis_distance=MAX_AXIS_DISTANCE,
    verbose=True
)

elapsed = time.perf_counter() - t0
print(f"\n✓ Completado en {elapsed:.1f}s")
print(f"Árboles detectados: {seg_result.n_trees}")
print(f"Puntos asignados: {len(veg_normalized) - seg_result.unassigned_count:,}")
print(f"Puntos sin asignar: {seg_result.unassigned_count:,}")

Segmentando árboles...
Input: 11,620,517 points
Step 1: Voxelizing...
  Voxelized to 4,422,247 voxels
Step 2: Extracting stripe (2.0-7.0m)...
  Stripe: 903,270 voxels
Step 3: Computing verticality...
  High verticality (>0.7): 473,247 voxels
Step 4: Clustering stems...
  Found 311 stem clusters
Step 5: Computing tree axes...
  Detected 142 tree axes
Step 6: Assigning points to trees...
  Assigned: 7,264,600 (62.5%)
  Unassigned: 4,355,917 (37.5%)

=== Summary ===
Trees detected: 142
  Tree 0: 54,221 pts, H=9.0m, dev=3.9°
  Tree 1: 55,762 pts, H=9.0m, dev=8.6°
  Tree 2: 20,314 pts, H=9.0m, dev=33.9°
  Tree 3: 8,184 pts, H=9.0m, dev=21.8°
  Tree 4: 10,541 pts, H=8.9m, dev=17.8°
  Tree 5: 17,777 pts, H=9.0m, dev=15.9°
  Tree 6: 127,758 pts, H=9.0m, dev=4.3°
  Tree 7: 22,480 pts, H=8.7m, dev=38.5°
  Tree 8: 33,188 pts, H=8.8m, dev=69.5°
  Tree 9: 84,574 pts, H=9.0m, dev=14.3°
  Tree 10: 18,455 pts, H=5.5m, dev=58.5°
  Tree 11: 24,291 pts, H=8.9m, dev=15.3°
  Tree 12: 118,333 pts, H=9.0m, d

In [9]:
# Resumen por árbol
print("\n=== Resumen por Árbol ===")
print(f"{'ID':>4} {'Puntos':>12} {'Altura Max':>12} {'Desv. Eje':>10}")
print("-" * 42)
for info in seg_result.tree_info:
    print(f"{info.tree_id:>4} {info.n_points:>12,} {info.height_max:>10.1f}m {info.axis_deviation_deg:>9.1f}°")


=== Resumen por Árbol ===
  ID       Puntos   Altura Max  Desv. Eje
------------------------------------------
   0       54,221        9.0m       3.9°
   1       55,762        9.0m       8.6°
   2       20,314        9.0m      33.9°
   3        8,184        9.0m      21.8°
   4       10,541        8.9m      17.8°
   5       17,777        9.0m      15.9°
   6      127,758        9.0m       4.3°
   7       22,480        8.7m      38.5°
   8       33,188        8.8m      69.5°
   9       84,574        9.0m      14.3°
  10       18,455        5.5m      58.5°
  11       24,291        8.9m      15.3°
  12      118,333        9.0m       8.1°
  13       11,045        9.0m      28.9°
  14       22,605        8.9m       4.3°
  15       45,638        9.0m       4.6°
  16       24,329        8.9m      49.6°
  17       42,608        9.0m      16.4°
  18       35,575        9.0m      30.0°
  19       54,804        9.0m      22.9°
  20       29,847        8.9m      20.7°
  21       91,730        9.

### 7.2 Exportar con tree_id

In [12]:
# Exportar nube segmentada con tree_id como campo escalar
import laspy

OUTPUT_DIR = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/raw")
seg_file = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/raw/tree_segmented_3.laz")

# Crear archivo LAS
header = laspy.LasHeader(version="1.4", point_format=0)
las_out = laspy.LasData(header)

las_out.x = veg_normalized[:, 0]
las_out.y = veg_normalized[:, 1]
las_out.z = veg_normalized[:, 2]

# Agregar tree_id como campo extra
las_out.add_extra_dim(laspy.ExtraBytesParams(name="tree_id", type="int32", description="Tree ID"))
las_out.tree_id = seg_result.tree_ids

las_out.write(str(seg_file))
print(f"✓ Exportado: {seg_file.name} ({seg_file.stat().st_size / (1024**2):.1f} MB)")
print(f"  Campo escalar 'tree_id' incluido para visualizar en CloudCompare")

✓ Exportado: tree_segmented_3.laz (55.8 MB)
  Campo escalar 'tree_id' incluido para visualizar en CloudCompare


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 7.2b CHECKPOINT - Exportar ubicaciones de árboles
# ═══════════════════════════════════════════════════════════════
import os
import sys
import importlib
import numpy as np
import laspy
from pathlib import Path

# Agregar módulo al path
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

# Recargar módulo para obtener versión actualizada
import src.core.segmentation as seg_module
importlib.reload(seg_module)
from src.core.segmentation import export_tree_locations, TreeSegmentationResult, TreeInfo

# Cargar archivo segmentado
SEG_FILE = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/raw/tree_segmented_1.laz")

las = laspy.read(str(SEG_FILE))
xyz = np.column_stack([las.x, las.y, las.z])
tree_ids = las.tree_id

print(f"✓ Cargados {len(xyz):,} puntos")
print(f"  Árboles únicos: {len(np.unique(tree_ids[tree_ids >= 0]))}")

# Reconstruir TreeInfo desde los datos
unique_ids = np.unique(tree_ids[tree_ids >= 0])
tree_info = []
for tid in unique_ids:
    mask = tree_ids == tid
    pts = xyz[mask]
    tree_info.append(TreeInfo(
        tree_id=int(tid),
        centroid=np.mean(pts, axis=0),
        n_points=int(np.sum(mask)),
        height_max=float(np.max(pts[:, 2])),
        height_min=float(np.min(pts[:, 2])),
        axis_direction=np.array([0, 0, 1]),
        axis_deviation_deg=0.0
    ))

seg_result = TreeSegmentationResult(
    tree_ids=tree_ids,
    n_trees=len(tree_info),
    unassigned_count=int(np.sum(tree_ids == -1)),
    tree_info=tree_info
)

# Exportar ubicaciones
locations_file = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/raw/tree_locations.txt")
tree_locations = export_tree_locations(seg_result, locations_file)

print(f"\n✓ Exportado: {locations_file.name}")
print(f"  {len(tree_locations)} ubicaciones")
print(f"  En CloudCompare: File > Open > ASCII cloud")

✓ Exportado: tree_locations.txt
  142 ubicaciones de árboles
  Formato: X Y Z tree_id height n_points

  En CloudCompare: File > Open, seleccionar como 'ASCII cloud'


### 7.3 Visualización por Árbol (Open3D)

In [ ]:
import open3d as o3d
import numpy as np

# Generar colores únicos por árbol
np.random.seed(42)
n_trees = seg_result.n_trees + 1  # +1 para no asignados
tree_colors = np.random.rand(n_trees, 3)
tree_colors[0] = [0.5, 0.5, 0.5]  # Gris para no asignados (si tree_id == -1)

# Asignar colores
point_colors = np.zeros((len(veg_normalized), 3))
for i, tid in enumerate(seg_result.tree_ids):
    if tid >= 0:
        point_colors[i] = tree_colors[tid + 1]
    else:
        point_colors[i] = tree_colors[0]

# Crear nube
pcd_seg = o3d.geometry.PointCloud()
pcd_seg.points = o3d.utility.Vector3dVector(veg_normalized)
pcd_seg.colors = o3d.utility.Vector3dVector(point_colors)

print(f"Nube segmentada: {len(pcd_seg.points):,} puntos, {seg_result.n_trees} árboles")

In [ ]:
o3d.visualization.draw_geometries([pcd_seg], window_name=f"Segmentación: {seg_result.n_trees} árboles", width=1280, height=720)

---
## 8. Próximos Pasos

- **Brick 8:** Análisis por árbol (DBH, altura, sweep, ramas)
- **Brick 9:** Detección de bifurcaciones
- **Brick 10:** Clasificación HQP de ramas y spikes